In [64]:
import json
import re

In [65]:
with open('./data/20_148_2024-01-19.json') as f:
    d = json.load(f)
    text = d['text']

In [71]:
name_match = r'((?:Dr\.\s)?(?:\w+(?:-\w+)?\s)+\w+(?:-\w+)?|\w+(?:\s\w+)*)'
partei_match = r'(CDU\/CSU|BÜNDNIS\s*90\/DIE\s*GRÜNEN|FDP|AfD|SPD)' # Parteilos + alte Parteien fehlen!
kommentar_match = rf'{name_match}\s+\[{partei_match}\]: (.*?)[–\)]'
beifall_match = r'[(?:–\s)\(]Beifall .*?[–\)]'
#beifall_match1 = r'\(Beifall .*?[–\)]'
zuruf_match = r'[(?:–\s)\(]Zuruf .*?[–\)]'



In [68]:
speaker_match = rf'{name_match}\s\({partei_match}\):'

speeches_raw = re.split(speaker_match, text)[1:]
speeches = []
for i in range(0, len(speeches_raw), 3):
    speech = {
        'redner':{
            'name':speeches_raw[i].strip(),
            'party':speeches_raw[i+1].strip()
        },
        'text': speeches_raw[i+2].strip()
        #'text': re.split(r'\n\n',speeches_raw[i+2])[0].strip() # cut off speakers without present party affiliation
    }
    speeches.append(speech)



In [69]:
print(speeches[5]['text'])

Sehr geehrte Frau Präsidentin! Werte Kolleginnen und Kollegen! Nachhaltigkeit bedeutet, die Bedürfnisse der Gegenwart so zu befriedigen, dass die Möglichkeiten künftiger Generationen nicht eingeschränkt werden. Nachhaltigkeit bedeutet aber auch, Wirtschaft, Soziales und Umweltschutz unter einen Hut zu bringen. Denn nur, wenn wir diesen Dreiklang auch wirklich zusammendenken, nur wenn wir es schaffen, wirtschaftlich zu wachsen und gleichzeitig Ressourcen zu schonen, können wir widerstandsfähiger sein.
Für unseren Wohlstand brauchen wir auch weiterhin Wachstum. Wir müssen dieses Wachstum aber nachhaltiger gestalten. Und genau das schafft die Ampel nicht, meine Damen und Herren. Als eines der wirtschaftlich leistungsfähigsten Länder der Welt steht Deutschland im Vergleich zu anderen Ländern viel schlechter da. Andere Staaten Europas und der Welt performen deutlich besser und verzeichnen trotz Krisen noch ein Wirtschaftswachstum. Bei uns in Deutschland schlägt die Rezession aber voll zu. U

In [48]:
speech_text =  speeches[5]['text']
for speech in speeches:
    comments = []
    for match in re.finditer(kommentar_match,speech['text']):
        comment = {
            'commentator': {
                'name': match.group(1),
                'party': match.group(2)
            },
            'text': match.group(3),
            'preceding_context': speech_text[:match.start()] 
        }
        comments.append(comment)
    speech['comments'] = comments


In [ ]:
json_string = json.dumps(speeches, indent=4) 

# Write JSON string to a file
with open("parsed_example.json", "w") as json_file:
    json_file.write(json_string)

# List of Abgeordnete

In [20]:
import xml.etree.ElementTree as ET

In [21]:

tree = ET.parse('./data/MDB_STAMMDATEN.XML')
root = tree.getroot()

In [26]:
bt_list = []
for mdb in root.findall('MDB'):
    dict = {
        'last_name':mdb.find('.//NACHNAME').text,
        'first_name':mdb.find('.//VORNAME').text,
        'anrede':mdb.find('.//ANREDE_TITEL').text,
        'party':mdb.find('.//PARTEI_KURZ').text,
        'election_period':mdb.find('.//WP').text

    }
    bt_list.append(dict)

    

In [27]:
bt_list

[{'last_name': 'Abelein',
  'first_name': 'Manfred',
  'anrede': 'Dr.',
  'party': 'CDU',
  'election_period': '5'},
 {'last_name': 'Achenbach',
  'first_name': 'Ernst',
  'anrede': 'Dr.',
  'party': 'FDP',
  'election_period': '3'},
 {'last_name': 'Ackermann',
  'first_name': 'Annemarie',
  'anrede': None,
  'party': 'CDU',
  'election_period': '2'},
 {'last_name': 'Ackermann',
  'first_name': 'Else',
  'anrede': 'Dr.',
  'party': 'CDU',
  'election_period': '11'},
 {'last_name': 'Adam',
  'first_name': 'Ulrich',
  'anrede': None,
  'party': 'CDU',
  'election_period': '12'},
 {'last_name': 'Adams',
  'first_name': 'Rudolf',
  'anrede': None,
  'party': 'SPD',
  'election_period': '5'},
 {'last_name': 'Adelmann',
  'first_name': 'Raban',
  'anrede': None,
  'party': 'CDU',
  'election_period': '3'},
 {'last_name': 'Adenauer',
  'first_name': 'Konrad',
  'anrede': 'Dr.',
  'party': 'CDU',
  'election_period': '1'},
 {'last_name': 'Adler',
  'first_name': 'Brigitte',
  'anrede': None,
 

In [2]:
mylist = root.xpath('//MDB/NAMEN/NAME/(VORNAME | NACHNAME | ANREDE_TITEL | AKAD_TITEL)/text()')
mylist

NameError: name 'root' is not defined

# For all Plenarprotokolle in a given folder

In [51]:
import os

In [61]:
data_path = './data/protokolle/'

<class 'list'>


In [63]:
for protokoll in os.listdir(data_path):
    with open(data_path+protokoll) as f:
        doc = json.load(f)
        text = doc['text']
    speaker_match = rf'{name_match}\s\({partei_match}\):'

    speeches_raw = re.split(speaker_match, text)[1:]
    speeches = []
    for i in range(0, len(speeches_raw), 3):
        speech = {  
            'redner':{
                'name':speeches_raw[i].strip(),
                'party':speeches_raw[i+1].strip()
            },
            #'text': speeches_raw[i+2].strip()
            'text': re.split(r'\n\n',speeches_raw[i+2])[0].strip() # cut off speakers like State ministers without party affiliation
        }

        comments = []
        for match in re.finditer(kommentar_match,speech['text']):
            comment = {
                'commentator': {
                    'name': match.group(1),
                    'party': match.group(2)
                },
                'text': match.group(3),
                'preceding_context': speech_text[:match.start()] 
            }
            comments.append(comment)
        speech['comments'] = comments

        speeches.append(speech)

    json_string = json.dumps(speeches, indent=4) 
    path = f'./data/parsed_comments/{doc['wahlperiode']}_{doc['dokumentnummer'].split(r'/')[1].zfill(3)}_{doc['datum']}_parsed.json'
    # Write JSON string to a file
    with open(path, "w") as json_file:
        json_file.write(json_string)   



In [5]:
kommentare = re.findall(kommentar_match, text)
kommentare

[('Dr.\xa0Anja Reinalter',
  'BÜNDNIS\xa090/DIE GRÜNEN',
  'Zum Thema Nachhaltigkeit!'),
 ('Dr.\xa0Anja Reinalter',
  'BÜNDNIS\xa090/DIE GRÜNEN',
  'Es geht nicht um Landwirte! Es geht um die Nachhaltigkeitsziele!'),
 ('Dr.\xa0Jan-Niclas Gesenhues',
  'BÜNDNIS\xa090/DIE GRÜNEN',
  'Auf die Schiene! Genau!'),
 ('Andreas Bleck',
  'AfD',
  'Ihr habt Deutschland abgewirtschaftet! Ihr werdet bei der nächsten Bundestagswahl abgestraft!\xa0'),
 ('Albrecht Glaser', 'AfD', 'Dreckiger Schmutz!'),
 ('Andreas Bleck', 'AfD', 'Das macht ihr! Noch nie ging es uns so schlecht!'),
 ('Andreas Bleck', 'AfD', 'Das entscheiden immer noch die Wähler!'),
 ('Andreas Bleck', 'AfD', 'Vor allem!'),
 ('Alexander Dobrindt', 'CDU/CSU', 'So ist es!'),
 ('Frank Bsirske', 'BÜNDNIS\xa090/DIE GRÜNEN', 'Blödsinn!'),
 ('Jakob Blankenburg',
  'SPD',
  'Wer war denn die letzten Jahre Landwirtschaftsminister?'),
 ('Bettina Hagedorn', 'SPD', 'Bei Ihnen passt gar nichts zusammen!'),
 ('Dr.\xa0Jan-Niclas Gesenhues',
  'BÜNDNIS

In [70]:
beifall_match = r'[(?:–\s)\(]Beifall .*?[–\)]'
#beifall_match1 = r'\(Beifall .*?[–\)]'

zuruf_match = r'[(?:–\s)\(]Zuruf .*?[–\)]'
re.findall(zuruf_match, text)

['(Zuruf des Abg. Friedrich Merz [CDU/CSU])',
 ' Zuruf des Abg. Alexander Dobrindt [CDU/CSU])',
 '(Zuruf vom BÜNDNIS\xa090/DIE GRÜNEN: Mein Gott!)',
 '(Zuruf des Abg. Dr.\xa0Rainer Kraft [AfD])',
 '(Zuruf des Abg. Dr.\xa0Rainer Kraft [AfD])',
 '(Zuruf des Abg. Dr.\xa0Rainer Kraft [AfD])',
 ' Zuruf von der AfD)',
 ' Zuruf vom BÜNDNIS\xa090/DIE GRÜNEN: Cheers!)',
 ' Zuruf von der SPD: Also, ich habe viel Vertrauen!)',
 '(Zuruf des Abg. Jakob Blankenburg [SPD])',
 '(Zuruf des Abg. Axel Müller [CDU/CSU])',
 '(Zuruf des Abg. Dr.\xa0Jan-Niclas Gesenhues [BÜNDNIS\xa090/DIE GRÜNEN])',
 '(Zuruf von der SPD)',
 '(Zuruf des Abg. Dr.\xa0Rainer Kraft [AfD])',
 '(Zuruf von der CDU/CSU: Was für eine Polemik!)',
 '(Zuruf von der AfD: Was hat das denn mit dem Thema zu tun?)',
 '(Zuruf vom BÜNDNIS\xa090/DIE GRÜNEN: Mein Gott!\xa0–',
 '(Zuruf von der AfD: Der hat doch schon geredet! Kommt jetzt was zum Thema?)',
 '(Zuruf vom BÜNDNIS\xa090/DIE GRÜNEN: Das sind auch Menschen!)',
 '(Zuruf von der CDU/CSU: D

In [8]:
with open("./testtext.txt" , 'w', encoding='utf8') as f:
    f.write(text)

In [ ]:
satzende_match = '(?<!\b(?:Dr|med|z\.B|etc)).\s+(?=[A-Z])'

# Sonderfälle

manche Redner wie z.B. amtierende Minister, Staatssekretäre etc. fallen aus dem Raster heraus und es steht keine Partei dahinter -> Wir wollen trotzdem die Parteien anmerken  
-> Abgleich mit XML-Stammdatenliste. Hier besonders angenehm: Schema: <p>"&lt;Vorname> &lt;Name>, &lt;Titel>:" </p> -> Das Ganze auch OHNE Doktortitel bei z.B. Dr. Robert Habeck -> Auslesen von Stammdaten xml